<a href="https://colab.research.google.com/github/hamnamuvees1720-oss/northstar-analytics/blob/main/notebooks/01_sql_in_r.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install necessary packages
install.packages(c('sqldf', 'dplyr', 'ggplot2'))

# Load libraries
library(sqldf)
library(dplyr)
library(ggplot2)

# This triggers the Python mount tool from within R
system("python3 -c 'from google.colab import drive; drive.mount(\"/content/drive\")'")

# Define the path
path <- "/content/drive/MyDrive/northstar_data/"

# Load files
orders     <- read.csv(paste0(path, "orders.csv"))
deliveries <- read.csv(paste0(path, "deliveries.csv"))
customers  <- read.csv(paste0(path, "customers.csv"))
drivers    <- read.csv(paste0(path, "drivers.csv"))
vehicles   <- read.csv(paste0(path, "vehicles.csv"))
hubs       <- read.csv(paste0(path, "hubs.csv"))
incidents  <- read.csv(paste0(path, "incidents.csv"))
complaints <- read.csv(paste0(path, "complaints.csv"))
app_events <- read.csv(paste0(path, "app_events.csv"))

# --- BEFORE: Unoptimised ---
# Capturing the time and the row count for the full join
t_slow <- system.time({
  slow <- sqldf(
    "SELECT d.*, o.pickup_zone, o.service_type, o.order_value
     FROM deliveries d
     JOIN orders o ON d.order_id = o.order_id
     WHERE d.delivery_status = 'Failed'")
})

# --- AFTER: Optimised ---
# Step 1: Pre-filter in R (This reduces the rows before SQL ever sees them)
failed_del <- deliveries[deliveries$delivery_status == 'Failed', ]

# Step 2: Join the significantly smaller dataset
t_fast <- system.time({
  fast <- sqldf(
    "SELECT d.*, o.pickup_zone, o.service_type, o.order_value
     FROM failed_del d
     JOIN orders o ON d.order_id = o.order_id")
})

# --- CALCULATION LOGIC ---
total_rows <- nrow(deliveries)
reduced_rows <- nrow(failed_del)
row_reduction_pct <- round((1 - (reduced_rows / total_rows)) * 100, 1)
speed_improvement <- round(((t_slow["elapsed"] - t_fast["elapsed"]) / t_slow["elapsed"]) * 100, 0)

# --- FORMATTED OUTPUT ---
cat(sprintf("BEFORE: %d rows processed in join → user: %.3fs elapsed: %.3fs\n",
            total_rows, t_slow["user.self"], t_slow["elapsed"]))

cat(sprintf("AFTER:  %d rows processed in join → user: %.3fs elapsed: %.3fs\n",
            reduced_rows, t_fast["user.self"], t_fast["elapsed"]))

cat(sprintf("Row reduction: %.1f%% (%d → %d)   |   Speed improvement: ~%d%% faster\n",
            row_reduction_pct, total_rows, reduced_rows, speed_improvement))







Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



BEFORE: 950 rows processed in join → user: 0.069s elapsed: 0.081s
AFTER:  132 rows processed in join → user: 0.064s elapsed: 0.076s
Row reduction: 86.1% (950 → 132)   |   Speed improvement: ~6% faster
